In [212]:
from tensorflow import keras # type: ignore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from datetime import datetime
import os
import pandas as pd
import numpy as np

In [213]:
#Opens a new dataframe with the Clean csv
cleancsv = pd.read_csv('CSV/CLEAN.csv')

In [214]:
#Convert data into Date time and create date filter
cleancsv['Date'] = pd.to_datetime(cleancsv['Date'])
cleancsv['Date'] = cleancsv['Date'] + pd.to_timedelta(cleancsv["Hr"], unit="h")
cleancsv.drop('Hr', axis=1, inplace=True)

"""
Use this in future if data set needs specific dates
prediction = data.loc{
    (untouched_csv['Date'] > datetime(x, x, x)) &
    (untouched_csv['Date'] < datetime(x, x, x,))
}
"""

"\nUse this in future if data set needs specific dates\nprediction = data.loc{\n    (untouched_csv['Date'] > datetime(x, x, x)) &\n    (untouched_csv['Date'] < datetime(x, x, x,))\n}\n"

In [215]:
#Create month colomn and restrict to only summer months
summer_mask = cleancsv["Date"].dt.month.isin([6, 7, 8])
cleancsv = cleancsv[summer_mask].reset_index(drop=True)

In [216]:
#Prepare colomns into variables
data_main_air_temp = cleancsv['Mainland Air Temp']
data_humidity_per = cleancsv['Humidity (%)']
data_wind_direction = cleancsv['Direction (A)']
data_wind_speed = cleancsv['Wind Speed (A)']
data_gusting = cleancsv['Gusting']
data_pressure = cleancsv['Atmospheric Pressure (IN)']
data_rainfall = cleancsv['Precipitation Rate']
data_bay_temp = cleancsv['Bay Temp']
data_salinity = cleancsv['Salinity']
data_lbi_temp = cleancsv['LBI Air Temp']
data_ocean_temp = cleancsv['Ocean Temp']
data_onshore_flag = cleancsv['Onshore']
data_upwelling_flag = cleancsv['upwelling_flag']

#saves all input data into one Numpy array
dataset = np.column_stack([
    data_main_air_temp.values,
    data_humidity_per.values,
    data_wind_direction.values,
    data_wind_speed.values,
    data_gusting.values,
    data_pressure.values,
    data_rainfall.values,
    data_bay_temp.values,
    data_salinity.values,
    data_lbi_temp.values,
    data_ocean_temp.values,
    #data_onshore_flag.values,
    data_upwelling_flag.values,
])

#Save output data into variables and reshape it to be a 2d array
output_data = data_onshore_flag.values
output_data = np.array(output_data).reshape(-1, 1)

In [217]:
#Length of training data
training_data_len = int(np.ceil(len(dataset) * 0.90)) #Use 90% of training data

In [218]:
#Scaler
scaler_x= StandardScaler()
scaler_y= StandardScaler()

scaledx = scaler_x.fit_transform(dataset)
scaledy = scaler_y.fit_transform(output_data)

training_data_x = scaledx[:training_data_len] #90% of all data
training_data_y = output_data[:training_data_len] #90% of all data

X_train, y_train = [], []

In [219]:
#Sliding window over last 24 hrs
for i in range(24, training_data_len):
    X_train.append(training_data_x[i-24:i, :])
    y_train.append(training_data_y[i, 0])

#Convert lists to arrays
X_train = np.array(X_train)
y_train = np.array(y_train).reshape(-1, 1) #1D Array

In [220]:
test_x = scaledx[training_data_len-24:]
X_test = []

#rebuild window
for i in range(24, len(test_x)):
    X_test.append(test_x[i-24:i, :])

X_test = np.array(X_test)   # (samples_test, 24, n_features)
print(X_test.shape)


(218, 24, 12)


In [221]:
#Build the model
model = keras.models.Sequential()

In [222]:
#Layer Zero (Shaping)
model.add(keras.layers.Input(shape=(X_train.shape[1], X_train.shape[2])))

In [223]:
#2nd Layer (LSTM)
model.add(keras.layers.LSTM(32, return_sequences=False))

In [224]:
#3rd Layer (Dense)
model.add(keras.layers.Dense(32, activation="relu"))

In [225]:
#Final Output Layer (Dense)
model.add(keras.layers.Dense(1))

In [226]:
#Put all the layers together
model.summary()
model.compile(optimizer="adam",
              loss="mae",
              metrics=[keras.metrics.RootMeanSquaredError()])

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_10 (LSTM)                  │ (None, 32)             │         5,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,849 (26.75 KB)

 Trainable params: 6,849 (26.75 KB)

 Non-trainable params: 0 (0.00 B)

In [227]:
#Train the model

#epochs = # of runs
#batch size = how much data is in each batch
training = model.fit(
X_train,
    y_train,
    epochs=44,
    batch_size=32,
    validation_split=0.1,
    shuffle=True
)

Epoch 1/44
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0686 - root_mean_squared_error: 0.1047 - val_loss: 0.0534 - val_root_mean_squared_error: 0.0653
Epoch 2/44
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0262 - root_mean_squared_error: 0.0349 - val_loss: 0.0440 - val_root_mean_squared_error: 0.0542
Epoch 3/44
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0194 - root_mean_squared_error: 0.0254 - val_loss: 0.0310 - val_root_mean_squared_error: 0.0381
Epoch 4/44
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0152 - root_mean_squared_error: 0.0202 - val_loss: 0.0272 - val_root_mean_squared_error: 0.0345
Epoch 5/44
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0128 - root_mean_squared_error: 0.0169 - val_loss: 0.0274 - val_root_mean_squared_error: 0.0347
Epoch 6/44
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0121 - root_mean_squared_error: 0.0157 - val_loss: 0.0264 - val_root_mean_squared_error: 0.0343
Epoch 7/44
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0112

In [228]:
prediction_scaled = model.predict(X_test)

# back to original units
prediction = scaler_y.inverse_transform(prediction_scaled)  


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


In [229]:
# rows of the original dataframe that correspond to X_test / prediction
test_index_start = training_data_len
test_index_end = training_data_len + prediction.shape[0]

test_df = cleancsv.iloc[test_index_start:test_index_end].copy()

# add predicted column
test_df["wind_pred"] = prediction.ravel()

#test_df.to_csv("CSV/predictions.csv", index=False)


In [230]:
# true wind speed for the same rows as predictions
y_test_true = cleancsv['Onshore'].iloc[test_index_start:test_index_end].values

mae = mean_absolute_error(y_test_true, prediction.ravel())
print("Test MAE (m/s):", mae)

Test MAE (m/s): 0.011140231043100357


In [231]:
print(prediction_scaled.shape)
print(prediction_scaled[:10].ravel())
print(prediction_scaled.mean(), prediction_scaled.std())

(218, 1)
[ 0.02161201  0.02051883  0.02134348  0.02467492  0.02580581  0.0241775
  0.01788583  0.00727836  0.00536598 -0.00158942]
0.007152014 0.0119226435


In [232]:

training.history['val_loss']

[0.053415507078170776,
 0.044024012982845306,
 0.030978940427303314,
 0.0271738450974226,
 0.02744397148489952,
 0.026382818818092346,
 0.02357042022049427,
 0.027226774021983147,
 0.022445496171712875,
 0.02360060065984726,
 0.023540208116173744,
 0.025401493534445763,
 0.023914510384202003,
 0.022122876718640327,
 0.02479531243443489,
 0.022115441039204597,
 0.023019468411803246,
 0.024399204179644585,
 0.022170666605234146,
 0.022287962958216667,
 0.02163366600871086,
 0.022223152220249176,
 0.0207440834492445,
 0.018869871273636818,
 0.018867941573262215,
 0.018075378611683846,
 0.02000259794294834,
 0.01934327930212021,
 0.01709054224193096,
 0.01658438704907894,
 0.016092121601104736,
 0.015584440901875496,
 0.018084479495882988,
 0.015957124531269073,
 0.015908056870102882,
 0.01374037005007267,
 0.015321501530706882,
 0.01380852796137333,
 0.014822948724031448,
 0.014728562906384468,
 0.013699589297175407,
 0.013695448637008667,
 0.012474344111979008,
 0.011702955700457096]

In [233]:
print("Wind speed mean/std:", output_data.mean(), output_data.std())
print("First 10 wind speeds:", output_data[:10].ravel())

Wind speed mean/std: 0.0 0.0
First 10 wind speeds: [0 0 0 0 0 0 0 0 0 0]


In [234]:
i0 = training_data_len   # first test index
print("Example window X[0] (first test sample):")
print(cleancsv.iloc[i0-24:i0][["Wind Speed (A)", "Mainland Air Temp", "Direction (A)"]].head(10))
print("Target at i0:", cleancsv["Wind Speed (A)"].iloc[i0])

Example window X[0] (first test sample):
      Wind Speed (A)  Mainland Air Temp  Direction (A)
1942             4.2               19.6              0
1943             4.6               18.3              0
1944             4.0               17.1             15
1945             3.9               16.1             15
1946             4.3               15.2             15
1947             3.8               14.7             15
1948             4.7               14.6             15
1949             5.7               14.1             15
1950             5.7               14.3             15
1951             6.1               14.9             15
Target at i0: 3.0
